In [1]:
import os, re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

schools = pd.read_csv("../data/processed/schools_with_athletics.csv")
sevp_all = pd.read_csv("../data/raw/sevp/sevp_certified_schools.csv")
sevp = sevp_all[sevp_all["f_visa"] == "Y"].copy()   # only F-1 matters for degree students
# Keep v1 results to compare what changed
old_path = "../data/processed/schools_with_sevp.csv"
old = pd.read_csv(old_path)[["unit_id", "sevp_certified"]] if os.path.exists(old_path) else None
print("Schools:", schools.shape, "| SEVP F-1 rows:", sevp.shape)

Schools: (3147, 44) | SEVP F-1 rows: (13267, 8)


In [2]:
ABBREV = {
    r"\bcoll\b": "college",
    r"\buniv\b": "university",
    r"\bcc\b": "community college",
    r"\bcomm\b": "community",
    r"\bctr\b": "center",
    r"\bmt\b": "mount",
    r"\bco\b": "county",      # "Butler Co. Community College"
    r"\bsaint\b": "st",       # makes "Saint" and "St." identical on BOTH sides ("State" untouched)
}

def norm(s):
    s = str(s).lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)               # punctuation -> space
    for pat, rep in ABBREV.items():
        s = re.sub(pat, rep, s)
    s = re.sub(r"\b(the|inc|llc)\b", " ", s)        # filler words
    s = re.sub(r"\bmain campus\b", " ", s)
    s = re.sub(r"^\s*(cuny|suny)\s+", " ", s)       # "CUNY LaGuardia..." -> "LaGuardia..."
    return re.sub(r"\s+", " ", s).strip()

def parent_name(s):
    # Branch campus -> parent institution: text before the LAST hyphen or slash.
    # "Penn State University-Penn State Harrisburg" -> "Penn State University"
    s = str(s)
    idx = max(s.rfind("-"), s.rfind("/"))
    return norm(s[:idx]) if idx > 0 else None

schools["name_n"] = schools["name"].apply(norm)
schools["parent_n"] = schools["name"].apply(parent_name)
schools["city_n"] = schools["city"].apply(norm)
sevp["school_n"] = sevp["school_name"].apply(norm)
sevp["campus_n"] = sevp["campus_name"].apply(norm)
sevp["city_n"] = sevp["city"].apply(norm)

In [3]:
sevp_by_state = {st: g for st, g in sevp.groupby("state")}   # blocking: same state only

def link(row):
    cand = sevp_by_state.get(row["state"])
    if cand is None:
        return pd.Series([None, None, None, 0.0])
    # Stage 1: exact normalized name vs SEVP school OR campus name
    hit = cand[(cand["school_n"] == row["name_n"]) | (cand["campus_n"] == row["name_n"])]
    if len(hit):
        h = hit.iloc[0]
        return pd.Series(["exact", h["school_name"], h["campus_name"], 100.0])
    # Stage 2: exact match on the PARENT institution name (branch campuses)
    if row["parent_n"]:
        hit = cand[(cand["school_n"] == row["parent_n"]) | (cand["campus_n"] == row["parent_n"])]
        if len(hit):
            h = hit.iloc[0]
            return pd.Series(["parent", h["school_name"], h["campus_name"], 100.0])
    # Stage 3: fuzzy. Same city: accept 85+. Different city: require 97+ (blocks Centra/Centura-type errors)
    s1 = cand["school_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x))
    s2 = cand["campus_n"].apply(lambda x: fuzz.token_sort_ratio(row["name_n"], x))
    score = np.maximum(s1, s2)
    best = score.idxmax()
    sc = float(score[best])
    same_city = cand.loc[best, "city_n"] == row["city_n"]
    method = "fuzzy" if ((sc >= 85 and same_city) or sc >= 97) else None
    b = cand.loc[best]
    return pd.Series([method, b["school_name"], b["campus_name"], round(sc, 1)])

schools[["sevp_match", "sevp_school", "sevp_campus", "sevp_score"]] = schools.apply(link, axis=1)
schools["sevp_certified"] = schools["sevp_match"].notna()

print("Match method:")
print(schools["sevp_match"].value_counts(dropna=False).to_string())
print("\nCertified by school type:")
print(pd.crosstab(schools["school_type"], schools["sevp_certified"], margins=True))
cols = ["name", "state", "sevp_school", "sevp_campus", "sevp_score"]
print("\n20 random FUZZY matches:")
fz = schools[schools["sevp_match"] == "fuzzy"]
print(fz.sample(min(20, len(fz)), random_state=1)[cols].to_string(index=False))
print("\n15 random PARENT matches:")
pm = schools[schools["sevp_match"] == "parent"]
print(pm.sample(min(15, len(pm)), random_state=1)[cols].to_string(index=False))

Match method:
sevp_match
exact     2276
None       664
parent     152
fuzzy       52
NaN          3

Certified by school type:
sevp_certified  False  True   All
school_type                      
2-year            461   889  1350
4-year            206  1591  1797
All               667  2480  3147

20 random FUZZY matches:
                                                      name state                                                       sevp_school                              sevp_campus  sevp_score
              Neighborhood Playhouse School of the Theater    NY                      Neighborhood Playhouse School of the Theatre NeighborhoodPlayhouseSchoolof theTheatre        97.5
             Embry-Riddle Aeronautical University-Prescott    AZ               Embry-Riddle Aeronautical University - Prescott, AZ      Embry-Riddle Flight Training Center        96.8
                     The University of Texas Permian Basin    TX                            The Univ of Texas of the Permian 

In [4]:
miss = schools[(~schools["sevp_certified"]) & (schools["pct_international"] >= 0.02)]
print("Likely linkage misses (2%+ international but not matched):", len(miss))
print(miss.nlargest(25, "pct_international")[
    ["name", "state", "pct_international", "sevp_school", "sevp_score"]
].round(3).to_string(index=False))

print("\nKansas 2-year schools:")
print(schools[(schools["state"] == "KS") & (schools["school_type"] == "2-year")][
    ["name", "sevp_certified", "sevp_match", "sevp_school", "sevp_score"]
].sort_values("name").to_string(index=False))

if old is not None:
    cmp = schools[["unit_id", "name", "state", "sevp_match", "sevp_school", "sevp_certified"]].merge(
        old.rename(columns={"sevp_certified": "sevp_certified_v1"}), on="unit_id"
    )
    gained = cmp[cmp["sevp_certified"] & ~cmp["sevp_certified_v1"]]
    lost = cmp[~cmp["sevp_certified"] & cmp["sevp_certified_v1"]]
    print(f"\nv1 -> v2: gained {len(gained)}, lost {len(lost)}")
    print("Lost (should be false positives like Centra College):")
    print(lost[["name", "state"]].to_string(index=False))

schools.to_csv("../data/processed/schools_with_sevp.csv", index=False)
print("\nSaved:", schools.shape)

Likely linkage misses (2%+ international but not matched): 92
                                                 name state  pct_international                                                         sevp_school  sevp_score
                The New England Conservatory of Music    MA              0.433                                            New England Conservatory        84.2
                    Lindsey Hopkins Technical College    FL              0.244                                          Southern Technical College        67.8
   Trine University-Regional/Non-Traditional Campuses    IN              0.236                                                    Trine University        63.7
                  Yeshiva Gedolah of Woodlake Village    NJ              0.218 Yeshiva Gedola of Woodlake Village DBA Yeshiva Ohr ZechariYaehshiva        66.7
                             American Islamic College    IL              0.188                                              Christian Life Coll

In [5]:
# Duke diagnostic: is Duke missing from the extraction, or mis-flagged F=N?
for term in ["duke", "buffalo", "laguardia", "highland", "butler"]:
    rows = sevp_all[sevp_all["school_name"].str.lower().str.contains(term, na=False) |
                    sevp_all["campus_name"].str.lower().str.contains(term, na=False)]
    print(f"\n'{term}' in SEVP extraction (all rows, incl. F=N): {len(rows)}")
    print(rows[["school_name", "campus_name", "f_visa", "m_visa", "city", "state", "parse_fix"]].head(10).to_string(index=False))


'duke' in SEVP extraction (all rows, incl. F=N): 2
                  school_name                   campus_name f_visa m_visa     city state parse_fix
Duke University & Health Sys. Duke University & Health Sys.      Y      N   Durham    NC      none
Duke University & Health Sys.    Duke University Marine Lab      Y      N Beaufort    NC      none

'buffalo' in SEVP extraction (all rows, incl. F=N): 9
                            school_name                             campus_name f_visa m_visa    city state     parse_fix
         Bryant & Stratton College, Inc     Bryant & Stratton College - Buffalo      Y      N BUFFALO    NY          none
    Buffalo Academy of the Sacred Heart     Buffalo Academy of the Sacred Heart      Y      N Buffalo    NY          none
                       Buffalo Seminary                        Buffalo Seminary      Y      N Buffalo    NY          none
           State University of New York           SUNY Buffalo State University      Y      N Buffalo    NY 